# Bangla ASR test: `Sanjidh090/moonshine-base-bn`

**Moonshine-Base** (61.5M params, encoder-decoder) adapted to Bengali by replacing its tokenizer
with the BanglaBERT WordPiece vocabulary, trained on the 882-hour Lipi-Ghor dataset. **License: MIT.**
Card (Lipi-Ghor test): **WER 21.5% / CER 10.8%** vs Hishab Conformer-Large zero-shot 30.5% / 18.2%.

**Gated (auto-approve):** open the model page, click *Agree and access repository*, then add a
Colab secret named `HF_TOKEN` (key icon on the left) with a READ token from
https://huggingface.co/settings/tokens.

> **RTF** = processing time / audio duration. Below 1.0 is faster than real time.
> Use **Runtime > Change runtime type > CPU** to measure CPU-server speed.

## 1. Settings

In [ ]:
BEAMS = 5                 # model card default; 1 = greedy (fastest)
THREADS = 0               # CPU threads, 0 = all cores
CHUNK_SEC = 20            # Moonshine is trained on short utterances; keep chunks short

AUDIO_SOURCE = "upload"   # "upload" or "drive"
DRIVE_AUDIO_DIR = "/content/drive/MyDrive/sample-audio"
HF_TOKEN = ""             # or use the Colab secret HF_TOKEN

## 2. Install dependencies (~1 minute)

In [ ]:
!apt-get -qq install -y ffmpeg libsndfile1 > /dev/null
!pip install -q -U transformers huggingface_hub librosa soundfile jiwer

## 3. Imports and environment check

In [ ]:
import os, sys, time, json, shutil, traceback
from pathlib import Path
import numpy as np, torch, librosa
if THREADS:
    torch.set_num_threads(THREADS)
print("python :", sys.version.split()[0])
print("torch  :", torch.__version__)
print("cores  :", os.cpu_count(), "| torch threads:", torch.get_num_threads())
!lscpu | grep -E "Model name|^CPU\(s\)"
import transformers
print("transformers:", transformers.__version__)

## 4. Download the model (~245 MB)

In [ ]:
os.environ["HF_HUB_DISABLE_XET"] = "1"   # Xet transfer can stall; plain HTTP is reliable
from huggingface_hub import snapshot_download
REPO_ID = "Sanjidh090/moonshine-base-bn"
MODEL_DIR = Path("/content/model")

token = HF_TOKEN or None
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception as e:
        print("No HF_TOKEN secret found:", e)

t0 = time.time()
snapshot_download(REPO_ID, local_dir=MODEL_DIR, token=token, allow_patterns=['*.json', '*.safetensors', 'README.md'])
print(f"Downloaded in {time.time()-t0:.0f}s")
!du -sh /content/model; ls -R /content/model | head -30

## 5. Runner (same code as `moonshine_base_bn/moonshine_bn.py`)

In [ ]:
"""
Runner for Sanjidh090/moonshine-base-bn (Moonshine-Base, 61.5M, BanglaBERT WordPiece vocab).

Follows the model card's inference recipe (beam search + repetition controls), with:
  * local, offline loading (no network)
  * max_new_tokens tied to the audio length, so a decoder that starts repeating
    on noise/silence cannot run away (an autoregressive model risk on call audio)
  * configurable beam size (1 = greedy, fastest)
"""

from pathlib import Path

import numpy as np
import torch

SR = 16000


class MoonshineBN:
    def __init__(self, model_dir, threads: int = 0, beams: int = 5):
        from transformers import AutoModelForSpeechSeq2Seq, AutoTokenizer

        path = Path(model_dir)
        if not (path / "model.safetensors").exists():
            raise FileNotFoundError(f"model.safetensors not found in {path}. Run download_model.py")
        if threads:
            torch.set_num_threads(threads)

        self.tokenizer = AutoTokenizer.from_pretrained(str(path), local_files_only=True)
        self.model = AutoModelForSpeechSeq2Seq.from_pretrained(
            str(path), dtype=torch.float32, local_files_only=True).eval()
        self.beams = beams
        # token ids, with the model card's verified defaults as fallback
        self.start_id = self.tokenizer.cls_token_id if self.tokenizer.cls_token_id is not None else 2
        self.eos_id = self.tokenizer.sep_token_id if self.tokenizer.sep_token_id is not None else 3
        self.pad_id = self.tokenizer.pad_token_id if self.tokenizer.pad_token_id is not None else 0

    @torch.inference_mode()
    def transcribe(self, y: np.ndarray) -> str:
        """y: 16 kHz mono float array (the caller chunks long audio)."""
        y = np.asarray(y, dtype=np.float32)
        if len(y) < 1600:  # < 0.1 s
            return ""
        rem = len(y) % 320
        if rem:
            y = np.concatenate([y, np.zeros(320 - rem, dtype=np.float32)])
        dur = len(y) / SR
        # ~1.3 tokens/word and fast Bangla speech is ~3-4 words/s -> 8 tokens/s is generous
        max_tokens = int(dur * 8) + 16
        ids = self.model.generate(
            torch.from_numpy(y).unsqueeze(0),
            max_new_tokens=max_tokens,
            num_beams=self.beams,
            no_repeat_ngram_size=3,
            repetition_penalty=1.2,
            decoder_start_token_id=self.start_id,
            pad_token_id=self.pad_id,
            eos_token_id=self.eos_id,
        )
        return self.tokenizer.decode(ids[0].tolist(), skip_special_tokens=True).strip()

## 6. Load the model

In [ ]:
t0 = time.time()
model = MoonshineBN(MODEL_DIR, threads=THREADS, beams=BEAMS)
LOAD_SEC = time.time() - t0
print(f"Model loaded in {LOAD_SEC:.1f}s | params {sum(p.numel() for p in model.model.parameters())/1e6:.1f}M")

## 7. Get your audio
* **upload**: select audio files, plus optional reference transcripts named like the audio
  (`call.mp3` + `call.txt`) to get WER/CER.
* **drive**: mounts Google Drive and reads `DRIVE_AUDIO_DIR`.

In [ ]:
AUDIO_DIR = Path("/content/sample-audio")
AUDIO_DIR.mkdir(exist_ok=True)
AUDIO_EXTS = {".wav", ".mp3", ".flac", ".ogg", ".m4a", ".opus", ".webm"}

if AUDIO_SOURCE == "upload":
    from google.colab import files
    for name, data in files.upload().items():
        (AUDIO_DIR / name).write_bytes(data)
elif AUDIO_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    for p in Path(DRIVE_AUDIO_DIR).iterdir():
        if p.suffix.lower() in AUDIO_EXTS | {".txt"}:
            shutil.copy(p, AUDIO_DIR / p.name)

AUDIO_FILES = sorted(p for p in AUDIO_DIR.iterdir() if p.suffix.lower() in AUDIO_EXTS)
print(f"{len(AUDIO_FILES)} audio file(s):")
for p in AUDIO_FILES:
    print(" -", p.name + (" (+ reference .txt)" if p.with_suffix(".txt").exists() else ""))

## 8. Run the test

In [ ]:
import jiwer
SR = 16000

def split_chunks(audio, chunk_sec):
    size = int(chunk_sec * SR)
    if size <= 0 or len(audio) <= size:
        return [audio]
    return [audio[i:i + size] for i in range(0, len(audio), size)]

results, errors = [], []
for f in AUDIO_FILES:
    print("=" * 80)
    print("FILE:", f.name)
    rec = {"file": f.name, "status": "ok"}
    try:
        audio, _ = librosa.load(str(f), sr=SR, mono=True)
        if audio.size == 0:
            raise ValueError("audio decoded to 0 samples")
        dur = len(audio) / SR
        chunks = split_chunks(audio, CHUNK_SEC)
        t0 = time.time()
        text = " ".join(t for t in (model.transcribe(c) for c in chunks) if t)
        infer = time.time() - t0
        rec.update(duration_sec=round(dur, 2), chunks=len(chunks), inference_sec=round(infer, 2),
                   rtf=round(infer / dur, 3), text=text)
        print(f"duration {dur:.2f}s | chunks {len(chunks)} | inference {infer:.2f}s | RTF {infer/dur:.3f}")
        print("TRANSCRIPT:\n" + (text or "<empty>"))
        ref_file = f.with_suffix(".txt")
        if ref_file.exists():
            ref = ref_file.read_text(encoding="utf-8").strip()
            rec.update(wer=round(jiwer.wer(ref, text), 4), cer=round(jiwer.cer(ref, text), 4))
            print(f"WER {rec['wer']:.2%} | CER {rec['cer']:.2%}")
    except Exception as e:
        rec.update(status="error", error=f"{type(e).__name__}: {e}")
        errors.append((f.name, traceback.format_exc()))
        print("ERROR:", rec["error"])
    results.append(rec)

## 9. Summary

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 120)
df = pd.DataFrame(results)
display(df[[c for c in df.columns if c not in ("text", "segments")]])
ok = df[df.status == "ok"]
if len(ok):
    print(f"Model load {LOAD_SEC:.1f}s | total audio {ok.duration_sec.sum():.1f}s | "
          f"inference {ok.inference_sec.sum():.1f}s | overall RTF {ok.inference_sec.sum()/ok.duration_sec.sum():.3f}")
print(f"{len(ok)} ok, {len(errors)} failed")

## 10. Errors (full tracebacks)

In [ ]:
if not errors:
    print("No errors")
for name, tb in errors:
    print("=" * 80)
    print("FILE:", name)
    print(tb)

## 11. Save and download results

In [ ]:
from datetime import datetime
out_json = f"/content/moonshine_base_bn_results_{datetime.now():%Y%m%d_%H%M%S}.json"
with open(out_json, "w", encoding="utf-8") as fh:
    json.dump({"model": REPO_ID, "beams": BEAMS, "chunk_sec": CHUNK_SEC, "threads": torch.get_num_threads(), "load_sec": round(LOAD_SEC, 2),
               "results": results, "errors": [{"file": n, "traceback": t} for n, t in errors]},
              fh, ensure_ascii=False, indent=2)
print("Saved", out_json)
from google.colab import files
files.download(out_json)